# 📘 Introduction to Asset Returns — Student Notebook

---

**🎯 Learning Objectives**

By the end of this lecture, you will be able to:

1. **Define and compute total returns** — From prices and dividends, using the standard formula
2. **Distinguish returns from excess returns** — And explain why the difference matters
3. **Interpret risk premiums and Sharpe ratios** — As measures of compensation for risk
4. **Annualize return statistics** — Converting daily or monthly data to annual terms
5. **Identify common pitfalls** — Unit errors, look-ahead bias, dividend handling, survivorship bias
6. **Audit AI-generated financial code** — Specify, implement, and validate return calculations

---

### How This Notebook Works

This is an **AI-assisted** notebook. Instead of pre-written code, you'll see:

- 📝 **Specifications** — what needs to happen, in plain English
- 🤖 **Suggested prompts** — try these in Gemini / ChatGPT / Claude
- 🔲 **Empty cells** — paste and run the AI-generated code here
- ✅ **Validation tasks** — checks to verify the code is correct

Code is available in hidden blocks if you need it, but **try the AI-first workflow.**

## 📋 Table of Contents

1. [What is a Stock Return?](#returns)
2. [What Can Go Wrong? The Pitfall Checklist](#pitfalls)
3. [Live Demo: Specify → Implement → Validate](#demo)
4. [Visualizing Returns](#viz)
5. [Excess Returns and Risk Premiums](#excess)
6. [The Data Unit Trap](#units)
7. [Sharpe Ratios and Cumulative Returns](#sharpe)
8. [A Brief Primer on Frequency](#frequency)
9. [Exercises](#exercises)
10. [Key Takeaways](#takeaways)

In [ ]:
#@title 🛠️ Setup: Run this cell first (click to expand)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully!")

---

## What is a Stock Return? <a id="returns"></a>

Imagine you buy a stock today for **\$100**. Tomorrow it's worth **\$105** and paid a **\$2 dividend**.

How do we measure what you earned?

### The Total Return Formula

$$R_t = \frac{P_t + D_t - P_{t-1}}{P_{t-1}} = \frac{P_t + D_t}{P_{t-1}} - 1$$

| Symbol | Meaning |
|--------|---------|
| $P_t$ | Price at end of period $t$ |
| $D_t$ | Dividend paid during period $t$ |
| $P_{t-1}$ | Price at start of period (end of $t-1$) |
| $R_t$ | Total return (as a decimal: 0.07 = 7%) |

> **💡 Key Insight: Why Returns, Not Prices?**
>
> Returns are the fundamental building block of finance because they are:
>
> - **Scale-free** — 10% on \$100 is comparable to 10% on \$1M
> - **Composable** — Multiply gross returns to get multi-period returns: $(1+R_1)(1+R_2)\cdots$
> - **Stationary** — Unlike prices (which drift upward), returns fluctuate around a stable mean
> - **Comparable** — You can compare Apple to Treasury bonds on the same scale

---

## 🛡️ What Can Go Wrong? The Pitfall Checklist <a id="pitfalls"></a>

When AI generates return calculations, it will often produce code that *runs without errors* but contains **silent bugs**. This checklist is your defense:

| | Pitfall | What Goes Wrong | 🔍 How to Detect |
|---|---------|----------------|-------------------|
| 1 | **Dividends ignored** | Ex-date price drops look like losses | Check: does your data include dividends? Verify a dividend day by hand |
| 2 | **Wrong lag** | Using $P_t$ instead of $P_{t-1}$ in denominator | First return should be NaN, not zero |
| 3 | **Data units** | RF in percent vs. decimal; annual vs. daily | Annualize and compare to known values (~5% Fed Funds) |
| 4 | **Stale prices** | Weekend/holiday prices create false zero returns | Check for suspiciously many exact zeros |
| 5 | **Look-ahead bias** | Using information not available at time $t$ | Signal must be lagged relative to the return it predicts |
| 6 | **Survivorship bias** | Only analyzing stocks that survived to today | Does the dataset include delisted firms? |

> **🤖 AI-Era Insight**
>
> In the age of AI-generated code, **your job is shifting from writing these calculations to specifying and auditing them.** This table is your checklist — print it out, tape it to your monitor, and use it every single time.

---

## 🔄 Live Demo: Specify → Implement → Validate <a id="demo"></a>

Instead of writing code from scratch, we'll practice the workflow you'll actually use in practice:

| Step | What You Do | Why It Matters |
|------|------------|----------------|
| **1. Specify** | Write a precise English description | Prevents misunderstandings and missing edge cases |
| **2. Implement** | Let AI generate the code (or write it yourself) | The easy part — syntax is cheap |
| **3. Validate** | Check edge cases, sanity-check outputs | This is where you earn your salary |

### Step 1: The Specification

> **📝 Specification**
>
> Load UNH (UnitedHealth) daily stock data from the provided URL. The CSV has columns `P` (adjusted closing price) and `D` (dividend, 0 on non-dividend days) with a `date` index. Compute daily total returns using $R_t = (P_t + D_t - P_{t-1}) / P_{t-1}$. Store results in a new column `ret`. The first observation should be NaN (no prior price available).

**Is this specification complete?** What's missing?
- ✅ Data source and format — specified
- ✅ Formula — specified
- ✅ Edge case (first row) — specified
- ⚠️ What if `D` has NaN instead of 0? — not addressed
- ⚠️ What if there are gaps in dates? — not addressed (the formula handles it via `.shift(1)`)

A complete spec addresses edge cases. But this is enough to start.

### Step 2: Implementation

> **🤖 Try This Prompt**
>
> Copy this into Gemini (or your preferred AI):
>
> *"Load a CSV from this URL into a pandas DataFrame with date as the index:
> `https://raw.githubusercontent.com/amoreira2/UG54/main/assets/data/UNH_data.csv`
> The CSV has columns `date`, `P` (price), and `D` (dividend).
> Then compute daily total returns using R_t = (P_t + D_t - P_{t-1}) / P_{t-1}.
> Store in a column called `ret`. Print the shape, date range, and show the first 10 rows."*

Paste the AI-generated code in the cell below and run it:

In [ ]:
# Paste your AI-generated code here and run it:



<details>
<summary>📦 Click to see reference code</summary>

```python
url = 'https://raw.githubusercontent.com/amoreira2/UG54/main/assets/data/UNH_data.csv'
df = pd.read_csv(url, parse_dates=['date'], index_col='date')

print(f"Data range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Columns: {list(df.columns)}")
print(f"Shape: {df.shape}")

# Compute total return
df['ret'] = (df['P'] + df['D'] - df['P'].shift(1)) / df['P'].shift(1)

df[['P', 'D', 'ret']].head(10)
```

</details>

### Step 3: Validate ✅

The code ran. It produced numbers. **But is it correct?**

Let's apply the pitfall checklist:

- ✅ **First return is NaN** — Correct, no prior price exists
- ✅ **Magnitudes look reasonable** — Daily returns of ~1-4% are typical for a single stock
- ⚠️ **Dividends** — Most days show D=0. Let's verify a dividend day by hand:

> **✅ Validation Task**
>
> Before trusting the output, verify it:
>
> 1. Is the first return NaN? (It should be — no prior price)
> 2. Find a day where a dividend was paid. Recompute the return **by hand** from P, D, and the prior price
> 3. Does your hand calculation match the computed return?
>
> **🤖 Try This Prompt:**
> *"Find all rows where D > 0 in df. Pick the first one. Get the previous day's price. Manually compute the return using the formula and compare to the computed value in df['ret']."*

In [ ]:
# Paste your validation code here:



<details>
<summary>📦 Click to see reference code</summary>

```python
div_days = df[df['D'] > 0]
print(f"Number of dividend days: {len(div_days)}")
display(div_days[['P', 'D', 'ret']].head(3))

# Manual verification
idx = div_days.index[0]
prev_idx = df.index[df.index.get_loc(idx) - 1]

p_t, d_t, p_prev = df.loc[idx, 'P'], df.loc[idx, 'D'], df.loc[prev_idx, 'P']
manual_ret = (p_t + d_t - p_prev) / p_prev

print(f"\n🔍 Manual check for {idx.date()}:")
print(f"   Manual:   {manual_ret:.6f}")
print(f"   Computed: {df.loc[idx, 'ret']:.6f}")
print(f"   ✅ Match: {abs(df.loc[idx, 'ret'] - manual_ret) < 1e-10}")
```

</details>

> **📌 Remember**
>
> Validation is not optional. Even correct-looking code can have subtle bugs. The habit of checking edge cases — first row, dividend days, extreme values — is what separates **good** quantitative work from **dangerous** quantitative work.

---

## 📊 Visualizing Returns <a id="viz"></a>

> **🤖 Try This Prompt**
>
> *"Create a figure with two subplots side by side (14×5 inches). Left: plot df['ret'] as a time series with a red dashed line at zero. Right: histogram of df['ret'] with 100 bins and a red dashed line at the mean. Use steelblue color."*

In [ ]:
# Paste your plotting code here:



<details>
<summary>📦 Click to see reference code</summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df.index, df['ret'], linewidth=0.5, alpha=0.8, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Daily Return')
axes[0].set_title('UNH Daily Returns', fontsize=13, fontweight='bold')

axes[1].hist(df['ret'].dropna(), bins=100, edgecolor='white', alpha=0.7, color='steelblue')
axes[1].axvline(df['ret'].mean(), color='red', linestyle='--', linewidth=2,
                label=f"Mean = {df['ret'].mean():.4%}")
axes[1].set_xlabel('Daily Return')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Returns', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()
```

</details>

> **🤖 Try This Prompt**
>
> *"Compute the daily mean and std of df['ret']. Annualize them (mean × 252, std × √252). Print a clean table with daily and annualized values."*

In [ ]:
# Paste your summary statistics code here:



<details>
<summary>📦 Click to see reference code</summary>

```python
mean_daily = df['ret'].mean()
std_daily = df['ret'].std()
mean_annual = mean_daily * 252
std_annual = std_daily * np.sqrt(252)

print("━" * 42)
print("  UNH Return Summary Statistics")
print("━" * 42)
print(f"  Daily mean return:      {mean_daily:>10.4%}")
print(f"  Daily std deviation:    {std_daily:>10.4%}")
print("  " + "─" * 38)
print(f"  Annualized mean:        {mean_annual:>10.2%}")
print(f"  Annualized volatility:  {std_annual:>10.2%}")
print("━" * 42)
```

</details>

> **💡 Key Insight: Annualization Rules**
>
> Financial data comes at different frequencies. To compare, we **annualize**:
>
> | Statistic | Rule | Daily → Annual | Monthly → Annual |
> |-----------|------|----------------|------------------|
> | Mean | Multiply by $N$ | × 252 | × 12 |
> | Volatility | Multiply by $\sqrt{N}$ | × $\sqrt{252}$ ≈ 15.87 | × $\sqrt{12}$ ≈ 3.46 |
> | Sharpe | Multiply by $\sqrt{N}$ | × $\sqrt{252}$ | × $\sqrt{12}$ |
>
> These are approximations (exact for log returns + iid). Standard in the industry — use them unless told otherwise.

---

## Excess Returns <a id="excess"></a>

A **10% return** sounds great — but what if risk-free bonds paid **8%**?

You only earned **2% extra** for taking on stock market risk. That's the excess return.

$$R^e_t = R_t - R^f_t$$

| Term | Meaning |
|------|---------|
| $R_t$ | Total return on the stock |
| $R^f_t$ | Risk-free rate (T-bill rate) |
| $R^e_t$ | **Excess return** — compensation for bearing risk |

---

## ⚠️ The Data Unit Trap <a id="units"></a>

> **⚠️ Caution**
>
> This is the **single most common error** in AI-generated financial code. The risk-free rate in Fama-French data is reported in specific units. An LLM will often skip the unit-check and silently use wrong units. **Your job is to catch it.**

### The Detective Work

The strategy: annualize the number and see if it matches reality.

> **🤖 Try This Prompt**
>
> *"Using pandas_datareader, load the Fama-French daily factors: `pdr.DataReader('F-F_Research_Data_Factors_Daily', 'famafrench', start='1900-01-01')[0]`. Print the date range and first few rows."*

In [ ]:
# Paste your code to load Fama-French data here:
import pandas_datareader.data as pdr



<details>
<summary>📦 Click to see reference code</summary>

```python
import pandas_datareader.data as pdr

start_date = '1900-01-01'
df_ff_daily = pdr.DataReader('F-F_Research_Data_Factors_Daily', 'famafrench', start=start_date)[0]
print(f"Data range: {df_ff_daily.index.min().date()} to {df_ff_daily.index.max().date()}")
print(f"Columns: {list(df_ff_daily.columns)}")
df_ff_daily.head()
```

</details>

### 🔍 The Detective Work

> **✅ Validation Task**
>
> The `RF` column contains the risk-free rate. But **what are the units?** This is the most common error in AI-generated finance code.
>
> **Do this yourself:**
> 1. Compute the mean daily RF for 2024: `df_ff_daily.loc['2024', 'RF'].mean()`
> 2. Annualize it: multiply by 252
> 3. Also try: multiply by 252 and divide by 100
> 4. Which result looks like a plausible annual interest rate? (~5% in 2024)

In [ ]:
# Your detective work here — figure out the units of RF:



<details>
<summary>📦 Click to see reference code</summary>

```python
rf_2024 = df_ff_daily.loc['2024', 'RF']

print(f"Mean daily RF (raw):       {rf_2024.mean():.4f}")
print(f"Annualized (× 252):        {rf_2024.mean() * 252:.2f}")
print(f"Annualized (× 252 / 100):  {rf_2024.mean() * 252 / 100:.4f}")

print(f"\nReality: Fed Funds ≈ 5% in 2024")
print(f"→ RF is in PERCENTAGE POINTS → divide by 100")
```

</details>

> **📌 Remember**
>
> This detective work — *annualizing a number and checking if it matches reality* — is a fundamental sanity check. An LLM will often skip this step and silently use wrong units. Your job is to catch it.
>
> **Rule of thumb:** If your annualized risk-free rate is > 10%, something is wrong.

> **🤖 Try This Prompt**
>
> *"Convert `df_ff_daily['RF']` from percentage points to decimal by dividing by 100. Store as `rf_daily`. Set the index name to 'date'. Merge with `df` on the date index using a left join. Then compute excess returns as `ret - rf_daily`. Print a comparison table of mean and std for total vs excess returns."*

In [ ]:
# Paste your merge + excess return code here:



<details>
<summary>📦 Click to see reference code</summary>

```python
df_ff_daily['rf_daily'] = df_ff_daily['RF'] / 100
df_ff_daily.index.name = 'date'

df_merged = df.merge(df_ff_daily[['rf_daily']], left_index=True, right_index=True, how='left')
df_merged['ret_excess'] = df_merged['ret'] - df_merged['rf_daily']

print("━" * 50)
print("  Return vs. Excess Return")
print("━" * 50)
print(f"  Mean total return:     {df_merged['ret'].mean():>12.4%}")
print(f"  Mean excess return:    {df_merged['ret_excess'].mean():>12.4%}")
print(f"  Mean RF (daily):       {df_merged['rf_daily'].mean():>12.4%}")
print("  " + "─" * 46)
print(f"  Std total return:      {df_merged['ret'].std():>12.4%}")
print(f"  Std excess return:     {df_merged['ret_excess'].std():>12.4%}")
print("━" * 50)
```

</details>

> **💡 Key Insight**
>
> The standard deviations are **nearly identical**. Why? Because $R^f$ is nearly constant day-to-day. Subtracting a constant shifts the mean but barely touches the volatility.
>
> But wait — the risk-free rate is *risk-free*. Should it affect the variance **at all**? (Think about this!)

### Excess Returns as Long-Short Strategies

Excess returns have an elegant interpretation as **self-financed trades**:

$$w \cdot R^e = w \cdot R + (-w) \cdot R^f = w(R - R^f)$$

- **Long** \$1 of the risky asset → earn $R$
- **Borrow** \$1 at the risk-free rate → pay $R^f$
- **Net investment:** \$0 (self-financed!)
- **Profit:** $R - R^f$ = excess return

> **💡 Key Insight: Mortgages Are Leveraged Long Positions**
>
> When you buy a house with a mortgage, you're running a *long-short strategy*:
> - **Long** real estate (the house)
> - **Short** cash (the mortgage loan)
> - Your "excess return" = house appreciation − mortgage rate
>
> If home prices only keep pace with your borrowing rate, you make nothing.

---

## 📈 Sharpe Ratios and Cumulative Returns <a id="sharpe"></a>

### Risk Premium and Sharpe Ratio

**Risk premium** = Expected excess return = $E[R - R^f]$

**Sharpe Ratio** = How much excess return *per unit of risk*:

$$\text{Sharpe Ratio} = \frac{E[R^e]}{\sigma(R^e)} = \frac{\text{Risk Premium}}{\text{Volatility}}$$

> **🤖 Try This Prompt**
>
> *"From `df_merged['ret_excess']`, compute the daily risk premium (mean), daily volatility (std), and annualized Sharpe ratio (daily mean/std × √252). Print all three in a clean table."*

In [ ]:
# Paste your Sharpe ratio code here:



<details>
<summary>📦 Click to see reference code</summary>

```python
rp_daily = df_merged['ret_excess'].mean()
vol_daily = df_merged['ret_excess'].std()

rp_annual = rp_daily * 252
vol_annual = vol_daily * np.sqrt(252)
sharpe_annual = (rp_daily / vol_daily) * np.sqrt(252)

print("━" * 42)
print("  UNH Risk Premium & Sharpe Ratio")
print("━" * 42)
print(f"  Annual risk premium:    {rp_annual:>10.2%}")
print(f"  Annual volatility:      {vol_annual:>10.2%}")
print(f"  Annual Sharpe ratio:    {sharpe_annual:>10.2f}")
print("━" * 42)
```

</details>

> **📌 Remember: Sharpe Ratio Benchmarks**
>
> | Sharpe | Interpretation |
> |--------|---------------|
> | < 0.3 | Weak risk-adjusted returns |
> | 0.3 – 0.5 | Decent (S&P 500 historical range) |
> | 0.5 – 1.0 | Good |
> | > 1.0 | Exceptional (and rare — check your math!) |
>
> **Why do we care?** The Sharpe ratio determines *optimal portfolio weights*. We'll see this when we study mean-variance optimization.

> **🤖 Try This Prompt**
>
> *"Compute cumulative returns for UNH (`(1 + df_merged['ret']).cumprod()`) and the risk-free rate (`(1 + df_merged['rf_daily']).cumprod()`). Plot both on the same chart with a log y-axis. Label them 'UNH' and 'Risk-Free'. Title: 'Cumulative Wealth: UNH vs Risk-Free'."*

In [ ]:
# Paste your cumulative returns plot here:



<details>
<summary>📦 Click to see reference code</summary>

```python
df_merged['cum_ret'] = (1 + df_merged['ret']).cumprod()
df_merged['cum_rf'] = (1 + df_merged['rf_daily']).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(df_merged.index, df_merged['cum_ret'], label='UNH', linewidth=2, color='steelblue')
ax.plot(df_merged.index, df_merged['cum_rf'], label='Risk-Free', linewidth=2,
        linestyle='--', color='gray')
ax.set_xlabel('Date')
ax.set_ylabel('Growth of $1 (log scale)')
ax.set_title('Cumulative Wealth: UNH vs Risk-Free', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=12)
ax.set_yscale('log')
plt.tight_layout()
plt.show()
```

</details>

> **💡 Key Insight**
>
> The **vertical gap** between the lines (on a log scale) is the cumulative excess return — the total compensation you received for bearing stock risk over decades. This is the risk premium, compounded.

---

## ⏱️ A Brief Primer on Frequency <a id="frequency"></a>

Financial data comes at different frequencies. The choice is **arbitrary** — trades happen every millisecond.

| Frequency | When to use | Observations per year |
|-----------|------------|----------------------|
| Daily | Short-horizon analysis, volatility estimation | ~252 |
| Monthly | Most academic research, factor models | 12 |
| Annual | Reporting, long-horizon comparisons | 1 |

### Exact Aggregation: Daily → Monthly or Annual

When you need *actual* annual returns (not approximations), you **compound**:

$$R_{\text{year}} = \prod_{t \in \text{year}}(1 + R_t) - 1$$

> **🐍 Python Insight: `groupby()` for Aggregation**
>
> The pandas `groupby()` method follows the **Split → Apply → Combine** pattern:
>
> ```python
> # Compound daily returns into annual returns
> annual_ret = (1 + daily_ret).groupby(daily_ret.index.year).prod() - 1
> ```
>
> | Step | What Happens |
> |------|-------------|
> | **Split** | Group data by year |
> | **Apply** | Multiply gross returns within each year |
> | **Combine** | One row per year |

> **🤖 Try This Prompt**
>
> *"Compound daily UNH returns into annual returns: use `(1 + df_merged['ret']).groupby(df_merged.index.year).prod() - 1`. Drop NaN. Then make a bar chart: green bars for positive years, red for negative, with a blue dashed line at the mean."*

In [ ]:
# Paste your annual returns code + bar chart here:



<details>
<summary>📦 Click to see reference code</summary>

```python
annual_ret = (1 + df_merged['ret']).groupby(df_merged.index.year).prod() - 1
annual_ret = annual_ret.dropna()

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#2ecc71' if x >= 0 else '#e74c3c' for x in annual_ret]
ax.bar(annual_ret.index, annual_ret, color=colors, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(annual_ret.mean(), color='steelblue', linestyle='--', linewidth=2,
           label=f'Mean = {annual_ret.mean():.1%}')
ax.set_xlabel('Year')
ax.set_ylabel('Annual Return')
ax.set_title('UNH Annual Returns', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()
```

</details>

---

## 📝 Exercises <a id="exercises"></a>

> **🤖 AI-Era Insight**
>
> These exercises emphasize **interpretation and auditing** over writing code from scratch. You may use AI to generate code, but **you are responsible for verifying correctness and explaining the results.**

### Exercise 1: Warm-Up — Basic Return Calculation

> **🔧 Exercise**
>
> A stock has the following prices over 4 days (no dividends):
>
> | Day | Price |
> |-----|-------|
> | 0 | \$50 |
> | 1 | \$52 |
> | 2 | \$48 |
> | 3 | \$51 |
>
> 1. Compute daily returns for Days 1, 2, and 3
> 2. Compute the cumulative return over the 3-day period
> 3. Verify: $(1+R_1)(1+R_2)(1+R_3) - 1$ equals the cumulative return

In [ ]:
# Your code here
prices = [50, 52, 48, 51]

# 1. Compute daily returns

# 2. Cumulative return

# 3. Verify compounding


<details>
<summary>💡 Click to see solution</summary>

```python
prices = [50, 52, 48, 51]

# Daily returns
R1 = (52 - 50) / 50   # = 0.04 = 4%
R2 = (48 - 52) / 52   # = -0.0769 = -7.69%
R3 = (51 - 48) / 48   # = 0.0625 = 6.25%

print(f"Day 1: {R1:.4%}")
print(f"Day 2: {R2:.4%}")
print(f"Day 3: {R3:.4%}")

# Cumulative return (simple)
cum = (51 - 50) / 50  # = 2%
print(f"\nCumulative return: {cum:.4%}")

# Verify via compounding
compounded = (1 + R1) * (1 + R2) * (1 + R3) - 1
print(f"Compounded: {compounded:.4%}")
print(f"Match: {abs(cum - compounded) < 1e-10}")
```

**Key point:** You can't just add returns! $(4\% - 7.69\% + 6.25\% = 2.56\% \neq 2\%)$. You must **compound**: multiply gross returns.

</details>

---

### Exercise 2: 🔍 Audit This Code

> **🤖 AI Audit Exercise**
>
> Your colleague asked an LLM to compute the Sharpe ratio for SPY. The LLM produced this code:
>
> ```python
> url2 = 'https://raw.githubusercontent.com/amoreira2/UG54/main/assets/data/df_SPYWMTJPM_data.csv'
> df2 = pd.read_csv(url2, parse_dates=['date'], index_col='date')
>
> # Annualized Sharpe ratio for SPY
> spy_mean = df2['SPY'].mean()
> spy_std = df2['SPY'].std() * np.sqrt(252)
> spy_sharpe = spy_mean / spy_std
> print(f"SPY Sharpe Ratio: {spy_sharpe:.2f}")
> ```
>
> **Find the bugs:**
> 1. What is wrong with the numerator?
> 2. This uses total returns, not excess returns. When would that matter a lot?
> 3. Fix the code and report the corrected Sharpe ratio.

In [ ]:
# Load the data
url2 = 'https://raw.githubusercontent.com/amoreira2/UG54/main/assets/data/df_SPYWMTJPM_data.csv'
df2 = pd.read_csv(url2, parse_dates=['date'], index_col='date')

# Run the BUGGY code first to see what it produces:
spy_mean = df2['SPY'].mean()
spy_std = df2['SPY'].std() * np.sqrt(252)
spy_sharpe = spy_mean / spy_std
print(f"BUGGY Sharpe: {spy_sharpe:.4f}")
print("Does this number make sense?")

# Now write your CORRECTED version below:



<details>
<summary>💡 Click to see solution</summary>

**Bug 1: Numerator not annualized.** `spy_mean` is the *daily* mean. The denominator is annualized (× √252) but the numerator is not (should be × 252). This makes the Sharpe ratio ~16× too small.

**Bug 2: Uses total returns, not excess returns.** For a proper Sharpe ratio, subtract RF first. This matters most when interest rates are high (e.g., the 1980s when RF ≈ 10%).

**Corrected code:**

```python
# Merge with risk-free rate
df2_merged = df2.merge(df_ff_daily[['rf_daily']], left_index=True, right_index=True, how='left')

# Excess returns
spy_excess = df2_merged['SPY'] - df2_merged['rf_daily']

# Correct Sharpe ratio
spy_sharpe = (spy_excess.mean() / spy_excess.std()) * np.sqrt(252)
print(f"Corrected SPY Sharpe: {spy_sharpe:.2f}")
```

</details>

---

### Exercise 3: 🤔 Interpretation — Tail Risk and VaR

> **🤔 Think and Code**
>
> You manage \$1 million invested in UNH.
>
> 1. Compute the **5th percentile** of daily returns (your "bad day" benchmark)
> 2. What **dollar amount** could you lose on a bad day?
> 3. Find the **worst single-day loss** in the historical sample
> 4. Compare the 1st percentile from your data vs. what a normal distribution predicts. Are stock returns normal?
> 5. **Write a one-paragraph memo** to a risk manager explaining why VaR based on the normal distribution might understate tail risk.
>
> **The deliverable is the memo, not the code.**

In [ ]:
# Your code here
portfolio_value = 1_000_000

# 1. 5th percentile

# 2. Dollar loss

# 3. Worst day

# 4. Normal distribution comparison (hint: use scipy.stats.norm.ppf)

# 5. Write your memo as a comment or markdown cell below


<details>
<summary>💡 Click to see solution</summary>

```python
from scipy import stats

portfolio_value = 1_000_000

# Risk metrics
pct_5 = df['ret'].quantile(0.05)
pct_1 = df['ret'].quantile(0.01)
worst = df['ret'].min()
worst_date = df['ret'].idxmin()

print(f"5th percentile:  {pct_5:.4%}  →  ${portfolio_value * abs(pct_5):,.0f} loss")
print(f"1st percentile:  {pct_1:.4%}  →  ${portfolio_value * abs(pct_1):,.0f} loss")
print(f"Worst day:       {worst:.4%}  →  ${portfolio_value * abs(worst):,.0f} loss  ({worst_date.date()})")
print(f"\nThe worst day was {worst/pct_5:.1f}× worse than the 5th percentile.")

# Normal distribution comparison
normal_1pct = stats.norm.ppf(0.01, df['ret'].mean(), df['ret'].std())
print(f"\nNormal model predicts 1st pct: {normal_1pct:.4%}")
print(f"Actual 1st pct:                {pct_1:.4%}")
print(f"Actual tail is {abs(pct_1)/abs(normal_1pct):.1f}× worse than normal predicts")
```

**Memo:**

> Stock returns exhibit "fat tails" — extreme losses occur more frequently than a normal distribution predicts. Our analysis of UNH shows the actual 1st percentile loss is significantly worse than the normal model. The worst historical day was several times beyond the 5th percentile. VaR estimates based on normal assumptions will systematically understate the probability and magnitude of large losses, particularly during market crises when correlations spike and losses cluster. A prudent risk framework should use historical (non-parametric) VaR or models that explicitly account for excess kurtosis.

</details>

---

### Exercise 4: 📝 The Specification Challenge — Three Levels of Prompting

> **🤖 AI Audit Exercise**
>
> Below are **three different prompts** for the same task: *compute the rolling 1-year Sharpe ratio for UNH*. Each produces different quality output from an LLM.
>
> **Your tasks:**
> 1. Read all three prompts. Which one would produce the most reliable code? Why?
> 2. Implement the task (using whichever prompt or your own code)
> 3. Plot the rolling Sharpe ratio
> 4. Find one period where the Sharpe was negative and explain what was happening

#### 🔴 Prompt A — Vague

> *"Compute the rolling Sharpe ratio for UNH and plot it."*

**What's missing?** Almost everything. What data? What window? Excess or total returns? Annualized or not? An LLM will make silent choices for all of these.

#### 🟡 Prompt B — Better

> *"Using daily UNH excess returns (already computed in `df_merged['ret_excess']`), compute the rolling 1-year Sharpe ratio. Use a 252-day window. Annualize by multiplying by √252. Plot the result."*

**Improved:** Specifies the data column, window, and annualization. **Still missing:** What if std is zero? What about the first 251 NaN values?

#### 🟢 Prompt C — Precise

> *"Using the column `df_merged['ret_excess']` (daily UNH excess returns), compute a rolling annualized Sharpe ratio as follows:*
> *1. Use a rolling window of 252 trading days.*
> *2. For each day t, compute: mean of excess returns over [t-251, t], and std of excess returns over the same window.*
> *3. Sharpe = (mean / std) × √252.*
> *4. Days before the window is full (first 251 rows) should be NaN.*
> *5. If std < 1e-10, set Sharpe to NaN (avoid division by zero).*
> *6. Store the result in `df_merged['rolling_sharpe']` and plot vs. date with a horizontal red line at zero."*

**This is a good specification.** It covers edge cases, specifies the output, and leaves no room for ambiguity.

In [ ]:
# Implement the rolling Sharpe ratio with a for-loop first
# This helps you see every step of the calculation
window = 252

# Your for-loop implementation here:



> **🐍 Python Insight: `rolling()` vs. for-loops**
>
> The for-loop above is transparent — you can see every step. But pandas provides `.rolling()` which is:
> - **~100× faster** on large datasets
> - **More concise** (2 lines vs. 15)
> - **The industry standard**
>
> Use the loop to *understand*; use `.rolling()` to *work*.
>
> ```python
> rolling_mean = df['ret_excess'].rolling(252).mean()
> rolling_std  = df['ret_excess'].rolling(252).std()
> sharpe = (rolling_mean / rolling_std) * np.sqrt(252)
> ```

In [ ]:
# Now implement using pandas .rolling() — compare to your for-loop
# Then plot the result

# Your .rolling() implementation here:


# Plot:


# Find negative-Sharpe periods:



---

## 🧠 Key Takeaways <a id="takeaways"></a>

1. **Total Return** — $R_t = (P_t + D_t - P_{t-1}) / P_{t-1}$ captures price change AND dividends

2. **Excess Return** — $R^e = R - R^f$ isolates compensation for bearing risk

3. **Risk Premium** — $E[R^e]$ is the expected reward for risk; estimated as the sample mean

4. **Sharpe Ratio** — Risk Premium / Volatility; the key measure of risk-adjusted performance

5. **Annualization** — Means × $N$, volatility × $\sqrt{N}$, Sharpe × $\sqrt{N}$ (where $N$ = periods per year)

6. **Always validate** — Check data units, verify edge cases, compare to known benchmarks

---

### 🤖 The New Skill

In an AI-assisted workflow, your value comes from three things:

| Skill | Example |
|-------|---------|
| **Precise specification** | "Use 252-day window on excess returns, annualize by √252" |
| **Careful validation** | Hand-check a dividend day, detect unit errors |
| **Financial judgment** | "A Sharpe of 3.0 can't be right — check the calculation" |

You no longer need to memorize `pd.merge()` syntax. But you **absolutely** need to know when a merge went wrong.

---

**Next:** We'll use returns to compare multiple assets — correlation, diversification, and the beginnings of portfolio theory.